Task 1

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

def bootstrap_ci(data, stat_func=np.mean, n_boot=10_000, ci_level=0.95):
    """
    Returns (lower, upper) percentile bootstrap CI.
    """
    data = np.array(data)
    n = len(data)
    boot_stats = []
    
    for _ in range(n_boot):
        resample = np.random.choice(data, size=n, replace=True)
        boot_stats.append(stat_func(resample))
        
    alpha = 1 - ci_level
    lower_p = (alpha / 2) * 100
    upper_p = (1 - alpha / 2) * 100
    
    lower = np.percentile(boot_stats, lower_p)
    upper = np.percentile(boot_stats, upper_p)
    
    return lower, upper


test_data = np.arange(1, 101)
lower, upper = bootstrap_ci(test_data)
print(f"Validation 95% CI for mean (1-100): [{lower:.2f}, {upper:.2f}]")

Task 2

In [ ]:
n_rows = 250
df = pd.DataFrame({
    'income': np.random.normal(loc=3500, scale=800, size=n_rows), 
    'purchased': np.random.binomial(n=1, p=0.4, size=n_rows)    
})

ci_mean = bootstrap_ci(df['income'], np.mean)
ci_median = bootstrap_ci(df['income'], np.median)
ci_prop = bootstrap_ci(df['purchased'], np.mean)

summary_data = {
    'Statistic': ['Mean (Income)', 'Median (Income)', 'Proportion (Purchased)'],
    'Lower Bound': [ci_mean[0], ci_median[0], ci_prop[0]],
    'Upper Bound': [ci_mean[1], ci_median[1], ci_prop[1]]
}

df_summary = pd.DataFrame(summary_data)
print(df_summary)

Task 3

In [ ]:
mean_val = df['income'].mean()
se_mean = st.sem(df['income'])
norm_mean_ci = st.t.interval(0.95, df=len(df['income'])-1, loc=mean_val, scale=se_mean)

p_hat = df['purchased'].mean()
n = len(df['purchased'])
se_prop = np.sqrt(p_hat * (1 - p_hat) / n)
z_star = 1.96
norm_prop_ci = (p_hat - z_star * se_prop, p_hat + z_star * se_prop)

comparison_df = pd.DataFrame({
    'Statistic': ['Mean', 'Proportion'],
    'Bootstrap Lower': [ci_mean[0], ci_prop[0]],
    'Bootstrap Upper': [ci_mean[1], ci_prop[1]],
    'Normal Lower': [norm_mean_ci[0], norm_prop_ci[0]],
    'Normal Upper': [norm_mean_ci[1], norm_prop_ci[1]]
})

print(comparison_df)

1.Yes, if the data is close to a normal distribution and the sample size ($n=250$) is large enough, both approaches will produce very close results. Divergence usually occurs when the data is very skewed or has many outliers.

Bootstrap is especially useful for the median. Because there is no simple and universal mathematical formula for calculating the standard error of the median,Bootstrap allows us to construct confidence intervals for any statistic without complicated formulas.